# 9. Estimating Token Costs: How Not to Bankrupt Yourself

Before you run an AI model on a big batch of data (500 customer reviews, a whole spreadsheet, a folder of documents) you should know roughly what it's going to cost. This notebook teaches you to **preview the cost before you spend anything**. This notebook talks to a model through OpenRouter, which bills per token. There is no free daily quota to fall back on if you skip this step. The habit matters more than it used to.

**Time**: ~15 minutes

**Cost to run this notebook**: one short trial call to OpenRouter. On the course default model that is a fraction of a cent. The character estimate and the `tiktoken` count still happen on your machine and do not call the API. The trial is there to measure how long the model's reply is.

> **How to run this notebook.** Open it in Cursor, VS Code, or Jupyter on your own computer. Create this week's environment once (`cd` into the week folder, then `uv venv` and `uv pip install -r requirements.txt`), select that `.venv` as the kernel, and keep your OpenRouter key in the project `.env` file. Full walkthrough: [Week 2's setup guide](lesson08_setup_guide.ipynb).

## Why This Matters

This course sends every model request through OpenRouter, which bills per token (Week 2's setup guide explains the setup). Every request costs real money, priced per **token**, not per request. A single test prompt costs a fraction of a cent. A loop that accidentally re-sends a 50-page document on every one of 10,000 iterations can cost real money before you notice, and there's no daily quota wall to stop it early the way a free tier used to.

The habit this notebook builds: **estimate first, run second.**

## What Is a Token?

AI models don't read text as words or letters. They read it as **tokens**, small chunks of text the model was trained to recognize. A token is often a word, but frequently it's a piece of a word.

For example, "marketing" might become two tokens: `market` + `ing`. As a rough rule for English text:

- **1 token ≈ 4 characters**
- **1 token ≈ ¾ of a word**

So a 100-word email is roughly 130–150 tokens. This isn't exact, but it's close enough to sanity-check a cost before you commit to running something.

**Two kinds of tokens, priced differently:**

- **Input tokens**: everything you send, your prompt, any pasted data, instructions.
- **Output tokens**: everything the model generates back. These are usually priced **2–6× higher** than input tokens, because generating text costs more than reading it.

If you're processing many items (rows in a spreadsheet, reviews, emails), your total cost is roughly:

```
number of items × (input tokens per item + output tokens per item) × price per token
```

## Quick Estimate: No API Call Needed

You don't need to call any API to get a ballpark. Run the cell below:

In [1]:
def estimate_tokens_quick(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English text."""
    return len(text) // 4

sample_review = "Great product, but shipping took way too long. Would still recommend to a friend."
print(f"Estimated tokens: {estimate_tokens_quick(sample_review)}")

Estimated tokens: 20


Use this the moment you're about to write a loop over a dataset, before you touch the API at all.

## Getting an Exact Count (no API charge)

When you want a precise number instead of a rough guess, use `tiktoken`. It is a local tokenizer: it splits text on your computer, without sending anything to OpenRouter.

This count is exact for OpenAI GPT-4o-family models. For the course default `qwen/qwen3-8b`, treat it as a close estimate, close enough to budget the **input** of a batch job.

`tiktoken` counts text you already have, such as your prompt. It cannot count the reply, because the reply does not exist yet. A single OpenRouter trial later in this notebook measures that reply.

Run this cell:

In [2]:
import pathlib
import subprocess
import sys

_cwd = pathlib.Path.cwd()
_week = "week_2"
if _cwd.name == _week and (_cwd / "requirements.txt").exists():
    _req_path = "requirements.txt"
elif (_cwd / _week / "requirements.txt").exists():
    _req_path = f"{_week}/requirements.txt"
elif (_cwd.parent / _week / "requirements.txt").exists():
    _req_path = f"../{_week}/requirements.txt"
else:
    _req_path = None

if _req_path is None:
    print("Couldn't find", _week + "/requirements.txt", "from:", _cwd)
    print("Set up the local environment first -- see Week 2's setup guide.")
else:
    # uv venvs do not include pip, so %pip fails with "No module named pip".
    subprocess.check_call(
        ["uv", "pip", "install", "-q", "-r", _req_path, "--python", sys.executable]
    )
    print("Libraries installed from", _req_path)

Note: you may need to restart the kernel to use updated packages.


d:\Personal project\ai4tm\.venv\Scripts\python.exe: No module named pip


In [8]:
import tiktoken

MODEL_NAME = "qwen/qwen3-8b"  # change this line to switch models

# o200k_base is the tokenizer used by GPT-4o and GPT-4o-mini.
encoding = tiktoken.get_encoding("o200k_base")

review = 'Analyze this customer review for sentiment: "Great product, slow shipping."'
token_count = len(encoding.encode(review))
print(f"Exact token count: {token_count}")
print("Model this count is calibrated for:", MODEL_NAME)

Exact token count: 14
Model this count is calibrated for: qwen/qwen3-8b


**If you see an error**: `tiktoken` is missing from the Python this notebook is using. Run the install cell above, or from a terminal: `uv pip install -r week_2/requirements.txt`. Then rerun the `import tiktoken` cell. Search "tiktoken encoding_for_model" if you want the encoding for a different OpenAI model family.

## Current Pricing (verify before you commit, these change often)

Prices are quoted **per 1 million tokens** and shift over time. Treat the table below as a starting point, not gospel. Check [openrouter.ai/models](https://openrouter.ai/models) before budgeting a large run. The numbers below are typical list prices for the cheap models this course uses; OpenRouter sometimes routes to a cheaper provider.

| Provider (via OpenRouter) | Model slug | Input $/1M tokens | Output $/1M tokens | Notes |
|---|---|---|---|---|
| Alibaba / Qwen | `qwen/qwen3-8b` | $0.12 | $0.46 | Course default |
| DeepSeek | `deepseek/deepseek-chat-v3.1` | check live | check live | Another slug that usually works in the same regions |
| Anthropic | `anthropic/claude-haiku-4.5` | $1.00 | $5.00 | Cheapest Claude option |
| Anthropic | `anthropic/claude-sonnet-4.5` | $3.00 | $15.00 | Mid-tier, higher quality |

**Official pricing pages** (bookmark these):
- OpenRouter: [openrouter.ai/models](https://openrouter.ai/models)
- OpenAI: [openai.com/api/pricing](https://openai.com/api/pricing/)
- Anthropic: [claude.com/pricing](https://claude.com/pricing)

⚠️ **Every request is billed.** OpenRouter has no daily free allowance hiding behind the key. Rate limits still apply. Set a credit limit on [openrouter.ai/settings/credits](https://openrouter.ai/settings/credits) before a large run.

## One trial, then the calculator

You can count the **input** before you spend anything (the cells above). You cannot know the **output** length until the model writes the answer. OpenRouter only reports that length after a real call. There is no free token-counting endpoint.

Run **one** trial on a typical item. Read `completion_tokens` from the reply. That number is the average output length you will assume for every item in the batch. The same reply also reports `prompt_tokens`, the billed input length for an item of this size. Put both numbers into the calculator and multiply by the number of items.

The trial is one billed call. On `qwen/qwen3-8b` it is a fraction of a cent. The calculator itself does not call the API again.

Pick an item that looks like the rest of the batch. A very short review will understate the cost. A very long one will overstate it. Use the same `max_tokens` cap here that you will use in the real loop, so the length you measure is the length you will allow.

In [9]:
from dotenv import load_dotenv
import os
from pathlib import Path
from openai import OpenAI

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / ".env").exists():
        load_dotenv(_candidate / ".env")
        break
else:
    load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError(
        "Could not load OPENROUTER_API_KEY. See Week 2's setup guide, then rerun this cell."
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

MODEL_NAME = "qwen/qwen3-8b"  # change this line to switch models
# Set this to None to leave the cap off. The ceiling cost then uses DEFAULT_MAX_OUTPUT_TOKENS.
MAX_OUTPUT_TOKENS = 80  # same cap you will use in the real loop
# OpenRouter lists 8,192 completion tokens for qwen/qwen3-8b when max_tokens is omitted.
# https://openrouter.ai/qwen/qwen3-8b
DEFAULT_MAX_OUTPUT_TOKENS = 8192

# One typical item. Swap this for a real row from your batch.
trial_item = "Great product, but shipping took way too long. Would still recommend to a friend."
trial_prompt = (
    "Analyze this customer review for sentiment. "
    "Reply with sentiment, confidence, and one key phrase.\n\n"
    f"Review: {trial_item}"
)

request = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": trial_prompt}],
}
if MAX_OUTPUT_TOKENS is not None:
    request["max_tokens"] = MAX_OUTPUT_TOKENS

response = client.chat.completions.create(**request)

usage = response.usage
trial_input_tokens = usage.prompt_tokens
avg_output_tokens = usage.completion_tokens  # this is the number the calculator needs
trial_cost = getattr(usage, "cost", None)
if trial_cost is None:
    trial_cost = (getattr(usage, "model_extra", None) or {}).get("cost")

print("Model:", MODEL_NAME)
print("Reply:", response.choices[0].message.content)
print("Input tokens for this item:", trial_input_tokens)
print("Output tokens (average for the batch):", avg_output_tokens)
details = usage.completion_tokens_details
reasoning_tokens = getattr(details, "reasoning_tokens", None) if details else None
if reasoning_tokens:
    print("Of which reasoning tokens:", reasoning_tokens)
    print("Reasoning tokens are billed. Use the output total above, not the length of the visible reply.")
if trial_cost is not None:
    print(f"This trial cost: ${trial_cost:.6f}")

Model: qwen/qwen3-8b
Reply: **Sentiment:** Positive  
**Confidence:** 85%  
**Key Phrase:** "Would still recommend to a friend"  

The review acknowledges a negative aspect (shipping time) but concludes with a strong positive endorsement, indicating overall satisfaction with the product.
Input tokens for this item: 46
Output tokens (average for the batch): 398
Of which reasoning tokens: 340
Reasoning tokens are billed. Use the output total above, not the length of the visible reply.
This trial cost: $0.000185


## Plug the trial into the calculator

The next cell prints **two** costs for the same batch:

- **Typical cost** uses `avg_output_tokens`, the reply length from the one trial.
- **Ceiling cost** uses `max_tokens`. That is the bill if every reply is as long as the cap. If `MAX_OUTPUT_TOKENS` is `None`, the ceiling uses `DEFAULT_MAX_OUTPUT_TOKENS` (8,192 for `qwen/qwen3-8b` on OpenRouter).

Change `NUM_ITEMS` to the size of your batch. Prices are dollars per 1 million tokens. Confirm them on [openrouter.ai/models](https://openrouter.ai/models) before a large run.

The model table reuses this trial's input length and only swaps the price. A different model may write a longer or shorter answer, and it may have a different default max. Change `MODEL_NAME` and `DEFAULT_MAX_OUTPUT_TOKENS`, then run the trial again.

In [10]:
def estimate_cost(
    num_items: int,
    input_tokens_per_item: int,
    avg_output_tokens: int,
    price_per_1m_input: float,
    price_per_1m_output: float,
) -> float:
    """Estimate total USD cost for a batch. Output length comes from one trial run."""
    input_cost = (num_items * input_tokens_per_item / 1_000_000) * price_per_1m_input
    output_cost = (num_items * avg_output_tokens / 1_000_000) * price_per_1m_output
    return input_cost + output_cost


def estimate_cost(
    num_items: int,
    input_tokens_per_item: int,
    avg_output_tokens: int,
    price_per_1m_input: float,
    price_per_1m_output: float,
) -> float:
    """Estimate total USD cost for a batch. Output length comes from one trial run."""
    input_cost = (num_items * input_tokens_per_item / 1_000_000) * price_per_1m_input
    output_cost = (num_items * avg_output_tokens / 1_000_000) * price_per_1m_output
    return input_cost + output_cost


NUM_ITEMS = 500
# qwen/qwen3-8b sample price. Confirm live before a large run.
PRICE_PER_1M_INPUT = 0.12
PRICE_PER_1M_OUTPUT = 0.46

if MAX_OUTPUT_TOKENS is None:
    cap_tokens = DEFAULT_MAX_OUTPUT_TOKENS
    cap_label = "model default max output"
else:
    cap_tokens = MAX_OUTPUT_TOKENS
    cap_label = "max_tokens"

cost_typical = estimate_cost(
    num_items=NUM_ITEMS,
    input_tokens_per_item=trial_input_tokens,
    avg_output_tokens=avg_output_tokens,
    price_per_1m_input=PRICE_PER_1M_INPUT,
    price_per_1m_output=PRICE_PER_1M_OUTPUT,
)
cost_at_cap = estimate_cost(
    num_items=NUM_ITEMS,
    input_tokens_per_item=trial_input_tokens,
    avg_output_tokens=cap_tokens,
    price_per_1m_input=PRICE_PER_1M_INPUT,
    price_per_1m_output=PRICE_PER_1M_OUTPUT,
)
print(f"One item: {trial_input_tokens} input tokens")
print(f"Typical cost ({avg_output_tokens} output tokens from the trial): ${cost_typical:.4f}")
print(f"Ceiling cost (every reply uses {cap_tokens} output tokens, {cap_label}): ${cost_at_cap:.4f}")
if avg_output_tokens > cap_tokens:
    print("The trial billed more output tokens than the cap. Thinking tokens can do that. Use the typical cost as the forecast.")

pricing = {
    "qwen/qwen3-8b": (0.12, 0.46),
    "deepseek/deepseek-chat-v3.1": (0.14, 0.28),
    "anthropic/claude-haiku-4.5": (1.00, 5.00),
    "anthropic/claude-sonnet-4.5": (3.00, 15.00),
}

print()
print(f"{'Model':<34}{'Typical':>12}{'At cap':>12}")
print("-" * 58)
for model_name, (price_in, price_out) in pricing.items():
    typical = estimate_cost(
        num_items=NUM_ITEMS,
        input_tokens_per_item=trial_input_tokens,
        avg_output_tokens=avg_output_tokens,
        price_per_1m_input=price_in,
        price_per_1m_output=price_out,
    )
    at_cap = estimate_cost(
        num_items=NUM_ITEMS,
        input_tokens_per_item=trial_input_tokens,
        avg_output_tokens=cap_tokens,
        price_per_1m_input=price_in,
        price_per_1m_output=price_out,
    )
    print(f"{model_name:<34}${typical:.4f}{at_cap:>12.4f}")

One item: 46 input tokens
Typical cost (398 output tokens from the trial): $0.0943
Ceiling cost (every reply uses 80 output tokens, max_tokens): $0.0212
The trial billed more output tokens than the cap. Thinking tokens can do that. Use the typical cost as the forecast.

Model                                  Typical      At cap
----------------------------------------------------------
qwen/qwen3-8b                     $0.0943      0.0212
deepseek/deepseek-chat-v3.1       $0.0589      0.0144
anthropic/claude-haiku-4.5        $1.0180      0.2230
anthropic/claude-sonnet-4.5       $3.0540      0.6690


The table prices 500 copies of the item you just measured. Output tokens usually make up most of the bill. If the output count is much larger than the short reply you can read, the model spent extra tokens thinking before it answered, and those tokens are billed too.

The real risk is an **unplanned loop**: retrying on every error without a cap, accidentally re-processing the same data twice, or pasting an entire multi-page document into every single request in a loop instead of once.

Estimate the total *before* you press run, especially when the item count is large or your prompts include a lot of pasted context.

## 5 Rules to Avoid Surprise Bills

1. **Measure one item, then estimate the loop.** Run one trial, read its output token count, and multiply item count × tokens × price *before* writing the `for` loop that calls the API.
2. **Test on the cheapest model first.** Prototype your prompt on the cheapest available model (`qwen/qwen3-8b`, a Flash-Lite slug, or Haiku). On a billed key none of these calls are free, but the cheapest models cost little enough to iterate on freely. Only upgrade to a pricier model if quality genuinely requires it.
3. **Cap your output length.** Most APIs accept a `max_tokens` (or similar) parameter. Set a sane ceiling so a model that starts rambling can't run up an unexpectedly large output bill.
4. **Don't re-send large context repeatedly.** If you're referencing the same big document across many calls, summarize it once instead of pasting the whole thing into every request.
5. **Set a spending limit in your provider dashboard.** OpenRouter lets you set a credit limit. OpenAI and Anthropic dashboards have the same idea if you ever use those keys directly. Turn one on for every key you use, it's the safety net for when your estimate is wrong.

## Where to Watch Your Real Spending

- **OpenRouter**: [openrouter.ai/activity](https://openrouter.ai/activity) for per-request usage, and [openrouter.ai/settings/credits](https://openrouter.ai/settings/credits) for a credit limit
- **OpenAI** (only if you also have a direct OpenAI key): [platform.openai.com/usage](https://platform.openai.com/usage)
- **Anthropic** (only if you also have a direct Anthropic key): [console.anthropic.com](https://console.anthropic.com)

Check these *before* a large run, not just after.

## You're Done

You now know how to:
- Estimate input token counts on your own computer, with no API call
- Measure the reply length with one OpenRouter trial (`completion_tokens`)
- Put that number into a cost calculator and project a batch before you loop
- Set guardrails (cheap model first, output cap, spending limit) so a mistake stays small

### Quick Reference

Keep this cell handy, copy it into any notebook where you're about to process a batch of data:

In [ ]:
# 1. One trial on a typical item.
# avg_output_tokens = response.usage.completion_tokens
# trial_input_tokens = response.usage.prompt_tokens
# If MAX_OUTPUT_TOKENS is None, the ceiling uses the model default (8192 for qwen/qwen3-8b).

# 2. Two costs. This does not call the API.
def estimate_cost(num_items, input_tokens_per_item, output_tokens,
                  price_per_1m_input, price_per_1m_output):
    input_cost = (num_items * input_tokens_per_item / 1_000_000) * price_per_1m_input
    output_cost = (num_items * output_tokens / 1_000_000) * price_per_1m_output
    return input_cost + output_cost

cap_tokens = MAX_OUTPUT_TOKENS if MAX_OUTPUT_TOKENS is not None else DEFAULT_MAX_OUTPUT_TOKENS
cost_typical = estimate_cost(NUM_ITEMS, trial_input_tokens, avg_output_tokens, PRICE_PER_1M_INPUT, PRICE_PER_1M_OUTPUT)
cost_at_cap = estimate_cost(NUM_ITEMS, trial_input_tokens, cap_tokens, PRICE_PER_1M_INPUT, PRICE_PER_1M_OUTPUT)

**The habit that matters most**: before a real dataset, run one typical item, read how many output tokens came back, then ask "how many items, at that length, at what price?" Do that multiplication before you do the loop.